# Segmentasi Instance Indoor — Training di Kaggle
**Seifeldein Akbar · NRP 5054241032 · digit terakhir 2 → variabel eksperimen: jumlah data train 25% / 50% / 100%**

Model yang dibandingkan (3 model, 2 keluarga arsitektur):

| Model | Keluarga | Mekanisme mask |
|---|---|---|
| Mask R-CNN ResNet-50 FPN | CNN, two-stage | RoIAlign → mask 28×28 per objek |
| YOLO26s-seg | CNN, one-stage, tanpa NMS | 32 prototype mask 160×160 (1/4 input 640) |
| RF-DETR-Seg Medium | Transformer (DINOv2), tanpa NMS | embedding piksel 1/4 resolusi input 432 (108×108) |

Urutan kerja:
1. **Bagian A (tanpa training):** instalasi → dataset v5 → audit + subset. Hasil audit dipakai untuk melengkapi hipotesis.
2. **Commit hipotesis ke Git**, lalu ubah `HIPOTESIS_SUDAH_DICOMMIT = True`. Hipotesis dan lampiran kelas sudah tercatat di logbook proyek.
3. **Bagian B:** smoke test → 9 training utama → evaluasi test asli → tabel & grafik.
4. **Bagian C:** test set foto HP sendiri, analisis kegagalan, demo.

Semua tahap berat punya sakelar `RUN_...`. Jika semua sakelar `False`, *Run All* hanya mendefinisikan fungsi.
Checkpoint dipilih dengan **mask AP validation**; split test tidak dipakai untuk memilih apa pun.

## 0. Pengaturan (ubah hanya sel ini)

In [ ]:
from pathlib import Path
import os, sys, json, time, math, random, shutil, csv, io, gzip, contextlib, datetime, platform, subprocess, hashlib, zipfile

# ---------- sakelar tahap ----------
RUN_INSTALL = True                 # pasang paket (Kaggle: Settings → Internet ON)
RUN_PREPARE = True                 # unduh/temukan dataset, audit, buat subset 25/50/100
HIPOTESIS_SUDAH_DICOMMIT = True    # hipotesis di LOGBOOK.md sudah di-commit (24 Nov 2026) sebelum training
RUN_SMOKE = True                   # 2 epoch per model pada 25% data (cek VRAM dan waktu)
RUN_TRAIN = False                  # training utama + evaluasi validation & test
RUN_SUMMARY = False                # tabel, grafik, ekspor hasil ringan
RUN_OWN_TEST = False               # evaluasi test set foto HP sendiri
RUN_DEMO = False                   # inferensi satu foto baru

# ---------- eksperimen ----------
NRP = '5054241032'
SEED = 42
PERCENTS = [25, 50, 100]
TRAIN_MODELS = ['yolo', 'maskrcnn', 'rfdetr']   # boleh dipecah per sesi Kaggle, mis. ['yolo']
TRAIN_PERCENTS = [25, 50, 100]
EPOCHS = 50
SMOKE_EPOCHS = 2
TAG = 't4_v5'
RUN_PROTOCOL = 'kaggle-t4-v5-1'
ACTIVE_RUNS = RUN_SMOKE or RUN_TRAIN or RUN_SUMMARY or RUN_OWN_TEST
PREVIOUS_RUN_INPUTS = []  # Isi path output versi Kaggle jika ingin membatasi input yang dipindai.
HAPUS_CHECKPOINT_LAIN = True   # simpan hanya checkpoint terpilih (batas disk Kaggle 20 GB)

# ---------- dataset ----------
# Folder yang berisi train/, valid/, test/ (masing-masing dengan _annotations.coco.json).
# Kaggle: dataset yang sudah diunggah. Lokal: folder ekspor Roboflow di samping notebook.
# Salin path persis dari panel Input Kaggle jika lokasi berbeda.
DATA_SOURCE = '/kaggle/input/datasets/seifeldeinakbar/panoptic-indoor'

# ---------- versi paket (jangan diubah di tengah eksperimen) ----------
PACKAGES = ['ultralytics==8.4.66', 'rfdetr[train]==1.12.1', 'pycocotools', 'roboflow']

IN_KAGGLE = 'KAGGLE_KERNEL_RUN_TYPE' in os.environ   # variabel ini hanya ada di notebook Kaggle
if ACTIVE_RUNS and not IN_KAGGLE:
    raise RuntimeError('Jalankan tahap eksperimen di Kaggle. Untuk demo lokal, matikan RUN_SMOKE, RUN_TRAIN, RUN_SUMMARY dan RUN_OWN_TEST.')
if IN_KAGGLE:
    os.environ['CUDA_VISIBLE_DEVICES'] = '0'  # Satu T4 meskipun accelerator menyediakan dua.
WORK = Path('/kaggle/working') if IN_KAGGLE else Path.cwd() / 'kerja'
RAW, PREP, RUNS, HASIL = WORK / 'data' / 'raw', WORK / 'data' / 'prepared', WORK / 'runs', WORK / 'hasil'
for _d in (RAW, PREP, RUNS, HASIL):
    _d.mkdir(parents=True, exist_ok=True)
try:
    display
except NameError:   # di luar Jupyter
    display = print
print('Kaggle:', IN_KAGGLE, '| folder kerja:', WORK)

## 1. Instalasi dan pencatatan lingkungan

In [ ]:
if ACTIVE_RUNS:
    gpu_check = subprocess.run(['nvidia-smi', '--query-gpu=name', '--format=csv,noheader'],
                               capture_output=True, text=True)
    if gpu_check.returncode or not any('T4' in line.split() for line in gpu_check.stdout.splitlines()):
        raise RuntimeError('Pilih GPU T4 (misalnya GPU T4 x2) di Settings Kaggle dan mulai sesi baru.')
if RUN_INSTALL:
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', *PACKAGES], check=True)

import numpy as np
import torch
from PIL import Image, ImageDraw, ImageOps
from pycocotools.coco import COCO
from pycocotools.cocoeval import COCOeval
from pycocotools import mask as mask_util


def pkg_version(name):
    try:
        import importlib.metadata as md
        return md.version(name)
    except Exception:
        return None


CUDA = torch.cuda.is_available()
GPU_NAME = torch.cuda.get_device_name(0) if CUDA else 'CPU'
VRAM_GB = torch.cuda.get_device_properties(0).total_memory / 1e9 if CUDA else 0.0
if ACTIVE_RUNS and (not CUDA or 'T4' not in GPU_NAME.split() or torch.cuda.device_count() != 1):
    raise RuntimeError(f'Diperlukan satu T4 yang terlihat oleh PyTorch. Terdeteksi {GPU_NAME}, CUDA={CUDA}.')
ENV = {
    'python': platform.python_version(), 'platform': platform.platform(), 'kaggle': IN_KAGGLE,
    'torch': torch.__version__, 'torchvision': pkg_version('torchvision'), 'cuda': torch.version.cuda,
    'gpu': GPU_NAME, 'vram_gb': round(VRAM_GB, 1), 'ultralytics': pkg_version('ultralytics'),
    'rfdetr': pkg_version('rfdetr'), 'pycocotools': pkg_version('pycocotools'), 'transformers': pkg_version('transformers'),
}
(HASIL / 'lingkungan.json').write_text(json.dumps(ENV, indent=2))
print(json.dumps(ENV, indent=2))
if not CUDA:
    print('PERINGATAN: GPU tidak terdeteksi. Kaggle: Settings → Accelerator → GPU T4.')

# Batch disesuaikan VRAM. Kaggle T4/P100 = 16 GB; RTX 4060 laptop = 8 GB.
BIG_GPU = VRAM_GB >= 14
WORKERS = 2 if os.name != 'nt' else 0
MODEL_CFG = {
    'maskrcnn': dict(batch=4 if BIG_GPU else 2, imgsz=640, optimizer='SGD momentum 0.9, wd 1e-4',
                     lr=0.02 * (4 if BIG_GPU else 2) / 16, warmup_iters=200, schedule='warmup linear + cosine',
                     val_every=1, augment='horizontal flip 0.5', workers=WORKERS),
    'yolo': dict(weights='yolo26s-seg.pt', batch=16 if BIG_GPU else 8, imgsz=640, optimizer='auto (default Ultralytics)',
                 augment='default Ultralytics', workers=4 if os.name != 'nt' else 0),
    'rfdetr': dict(variant='RFDETRSegMedium', batch=4 if BIG_GPU else 2, grad_accum=4 if BIG_GPU else 8, lr=1e-4,
                   imgsz=432, augment='default RF-DETR', workers=WORKERS),
}
print(json.dumps(MODEL_CFG, indent=2))

## 2. Dataset Panoptic Indoor Segmentation v5 (format COCO Segmentation)
Tambahkan ekspor COCO Segmentation v5 lewat Add Input. Isi DATA_SOURCE dengan path input tersebut. Sumber dan sidik jari anotasi disimpan bersama hasil.

In [ ]:
SPLITS = ('train', 'valid', 'test')


def find_coco_root(base):
    base = Path(base)
    if not base.exists():
        return None
    for ann in sorted(base.rglob('_annotations.coco.json')):
        root = ann.parent.parent
        if all((root / s / '_annotations.coco.json').exists() for s in SPLITS):
            return root
    return None


def get_dataset():
    root = find_coco_root(DATA_SOURCE)
    if root is None:
        raise FileNotFoundError(
            f'Ekspor v5 tidak ditemukan di {DATA_SOURCE}. Tambahkan dataset melalui Add Input '
            'dan isi DATA_SOURCE dengan path folder sumber yang berisi train, valid, dan test.'
        )
    return root


## 3. Audit dataset, subset 25/50/100, dan konversi format
* Ekspor asli tidak diubah. Gambar ditautkan (symlink/hardlink) ke folder kerja.
* Kategori root sintetis Roboflow (dipakai sebagai `supercategory`, tanpa anotasi) dibuang; kelas lain dipertahankan,
  termasuk nama salah ketik.
* Subset bersarang dengan seed 42: 25% ⊂ 50% ⊂ 100%, sama untuk ketiga model. Validation dan test tidak diperkecil.
* Statistik bentuk kelas (luas relatif, *fill ratio*) dihitung dari **train + valid saja**, tidak dari test.

In [ ]:
def load_json(p):
    return json.loads(Path(p).read_text(encoding='utf-8'))


def save_json(obj, p, indent=None):
    p = Path(p)
    p.parent.mkdir(parents=True, exist_ok=True)
    p.write_text(json.dumps(obj, ensure_ascii=False, indent=indent), encoding='utf-8')


@contextlib.contextmanager
def quiet():
    with contextlib.redirect_stdout(io.StringIO()):
        yield


def link_or_copy(src, dst):
    dst = Path(dst)
    if dst.exists():
        return
    dst.parent.mkdir(parents=True, exist_ok=True)
    for fn in (os.symlink, os.link):
        try:
            fn(os.path.abspath(src), dst)
            return
        except (OSError, NotImplementedError, AttributeError):
            pass
    shutil.copy2(src, dst)


def load_rgb(path, info=None):
    img = Image.open(path)
    if info is not None and img.size != (info['width'], info['height']):
        turned = ImageOps.exif_transpose(img)
        if turned.size != (info['width'], info['height']):
            raise ValueError(f'{path}: ukuran {img.size} tidak sama dengan anotasi {(info["width"], info["height"])}')
        img = turned
    return img.convert('RGB')


# 4 gambar di ekspor v5 tersimpan potret (480x640) tanpa tag EXIF, padahal anotasinya landscape (640x480).
# Dicek visual 24 Nov 2026 (182169107, 182169108, 182169109, 182169121): polygon cocok setelah Image.ROTATE_90
# (90 derajat berlawanan jarum jam). Gambar seperti ini disalin dalam keadaan sudah diputar; ekspor asli tidak diubah.
ROTASI_GAMBAR_TERTUKAR = Image.ROTATE_90


def orientation_fix(path, info):
    """None jika ukuran cocok; 'exif' jika cukup exif_transpose; 'rot90' jika lebar/tinggi tertukar."""
    img = Image.open(path)
    if img.size == (info['width'], info['height']):
        return None
    if ImageOps.exif_transpose(img).size == (info['width'], info['height']):
        return 'exif'
    if img.size == (info['height'], info['width']):
        return 'rot90'
    raise ValueError(f'{path}: ukuran {img.size} tidak cocok dengan anotasi {(info["width"], info["height"])}')


def place_image(src, dst, fix):
    if fix is None:
        link_or_copy(src, dst)
        return
    dst = Path(dst)
    if dst.exists():
        return
    dst.parent.mkdir(parents=True, exist_ok=True)
    img = Image.open(src)
    img = ImageOps.exif_transpose(img) if fix == 'exif' else img.transpose(ROTASI_GAMBAR_TERTUKAR)
    img.convert('RGB').save(dst, quality=95, subsampling=0)


def mask_area(ann, h, w):
    """Luas mask sebenarnya. Field 'area' ekspor Roboflow berisi luas bounding box, bukan luas mask."""
    seg = ann['segmentation']
    if isinstance(seg, list):
        rle = mask_util.merge(mask_util.frPyObjects(seg, h, w))
    elif isinstance(seg.get('counts'), list):
        rle = mask_util.frPyObjects(seg, h, w)
    else:
        rle = seg
    return float(mask_util.area(rle))


def real_categories(cocos):
    cats = cocos['train']['categories']
    ref = [(c['id'], c['name']) for c in cats]
    for s in SPLITS:
        if [(c['id'], c['name']) for c in cocos[s]['categories']] != ref:
            raise ValueError(f'Daftar kategori split {s} berbeda dengan train')
    annotated = {a['category_id'] for s in SPLITS for a in cocos[s]['annotations']}
    def is_root(c):
        is_parent = any(o.get('supercategory') == c['name'] and o['id'] != c['id'] for o in cats)
        return is_parent and c['id'] not in annotated
    real = [c for c in cats if not is_root(c)]
    names = [c['name'] for c in real]
    dup = sorted({n for n in names if names.count(n) > 1})
    if dup:
        raise ValueError(f'Nama kelas ganda: {dup}')
    return real, [c for c in cats if is_root(c)]


def polygons_of(ann, h, w):
    seg = ann.get('segmentation')
    if isinstance(seg, list):
        return [p for p in seg if len(p) >= 6]
    if isinstance(seg, dict):  # RLE → kontur
        import cv2
        rle = mask_util.frPyObjects(seg, h, w) if isinstance(seg.get('counts'), list) else seg
        m = mask_util.decode(rle).astype(np.uint8)
        cs, _ = cv2.findContours(m, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
        return [c.reshape(-1).astype(float).tolist() for c in cs if len(c) >= 3]
    return []


def yolo_line(cls, polys, w, h):
    from ultralytics.data.converter import merge_multi_segment
    if len(polys) > 1:
        s = np.concatenate(merge_multi_segment(polys), axis=0)
    else:
        s = np.array(polys[0], dtype=float).reshape(-1, 2)
    s = np.clip(s / np.array([w, h]), 0, 1).reshape(-1)
    return f'{cls} ' + ' '.join(f'{v:.6f}' for v in s)


def prepare(root, seed=SEED):
    root = Path(root)
    if (PREP / 'audit.json').exists():
        print('Dataset sudah disiapkan di', PREP)
        return load_json(PREP / 'audit.json')
    cocos = {s: load_json(root / s / '_annotations.coco.json') for s in SPLITS}
    real, roots = real_categories(cocos)
    real_ids = [c['id'] for c in real]
    yolo_idx = {cid: i for i, cid in enumerate(real_ids)}
    audit = {'sumber': str(root), 'tanggal': datetime.datetime.now().isoformat(timespec='seconds'),
             'kategori_root_dibuang': roots, 'jumlah_kelas': len(real), 'split': {}, 'masalah': [],
             'gambar_diperbaiki_orientasi': [],
             'catatan_area': 'field area diganti luas mask dari polygon (ekspor Roboflow berisi luas bounding box)'}
    hashes, good_anns, fixes = {}, {}, {}
    stats = {cid: {'id': cid, 'name': c['name'], 'area_rel': [], 'fill': []} for cid, c in zip(real_ids, real)}
    for s in SPLITS:
        imgs = cocos[s]['images']
        by_img = {}
        for a in cocos[s]['annotations']:
            by_img.setdefault(a['image_id'], []).append(a)
        n_bad = 0
        for info in imgs:
            path = root / s / info['file_name']
            if not path.exists():
                audit['masalah'].append(f'{s}: file hilang {info["file_name"]}')
                continue
            try:
                fix = orientation_fix(path, info)
                if fix:
                    fixes[(s, info['id'])] = fix
                    audit['gambar_diperbaiki_orientasi'].append(f'{s}/{info["file_name"]} ({fix})')
            except Exception as e:
                audit['masalah'].append(f'{s}: {e}')
            hashes.setdefault(__import__('hashlib').md5(path.read_bytes()).hexdigest(), []).append(f'{s}/{info["file_name"]}')
            kept = []
            for a in by_img.get(info['id'], []):
                if a['category_id'] not in yolo_idx:
                    audit['masalah'].append(f'{s}: anotasi {a["id"]} kategori {a["category_id"]} bukan kelas nyata')
                    n_bad += 1
                    continue
                polys = polygons_of(a, info['height'], info['width'])
                area = mask_area(a, info['height'], info['width']) if polys else 0.0
                if not polys or area <= 0:
                    n_bad += 1
                    continue
                a = {**a, 'area': area}
                kept.append((a, polys))
                st = stats[a['category_id']]
                st[f'{s}_inst'] = st.get(f'{s}_inst', 0) + 1
                st.setdefault(f'{s}_img', set()).add(info['id'])
                if s != 'test':
                    bw, bh = a['bbox'][2], a['bbox'][3]
                    st['area_rel'].append(area / (info['width'] * info['height']))
                    if bw * bh > 0:
                        st['fill'].append(area / (bw * bh))
            good_anns[(s, info['id'])] = kept
        audit['split'][s] = {'gambar': len(imgs), 'anotasi': len(cocos[s]['annotations']),
                             'anotasi_tidak_valid_dibuang': n_bad,
                             'gambar_tanpa_anotasi': sum(1 for i in imgs if not good_anns.get((s, i['id'])))}
    dups = [v for v in hashes.values() if len({x.split('/')[0] for x in v}) > 1]
    audit['duplikat_identik_antar_split'] = dups

    train_ids = sorted(i['id'] for i in cocos['train']['images'])
    order = train_ids[:]
    random.Random(seed).shuffle(order)
    subsets = {p: sorted(order[:math.floor(len(order) * p / 100)]) for p in PERCENTS}
    assert set(subsets[25]) <= set(subsets[50]) <= set(subsets[100])
    audit['subset'] = {f'p{p}': len(v) for p, v in subsets.items()}
    save_json({f'p{p}': v for p, v in subsets.items()}, PREP / 'subset_ids.json')

    # COCO per subset (dipakai Mask R-CNN, RF-DETR, dan evaluator) + YOLO
    names = {i: c['name'] for i, c in enumerate(real)}
    for p in PERCENTS:
        for s in SPLITS:
            keep = set(subsets[p]) if s == 'train' else {i['id'] for i in cocos[s]['images']}
            images = [i for i in cocos[s]['images'] if i['id'] in keep]
            anns = [a for i in images for a, _ in good_anns.get((s, i['id']), [])]
            out_dir = PREP / 'coco' / f'p{p}' / s
            for i in images:
                place_image(root / s / i['file_name'], out_dir / i['file_name'], fixes.get((s, i['id'])))
            save_json({**{k: v for k, v in cocos[s].items() if k not in ('images', 'annotations')},
                       'images': images, 'annotations': anns}, out_dir / '_annotations.coco.json')
            ysplit = f'train_p{p}' if s == 'train' else s
            if s != 'train' and p != PERCENTS[-1]:
                continue
            for i in images:
                place_image(root / s / i['file_name'], PREP / 'yolo' / 'images' / ysplit / i['file_name'],
                            fixes.get((s, i['id'])))
                lines = [yolo_line(yolo_idx[a['category_id']], polys, i['width'], i['height'])
                         for a, polys in good_anns.get((s, i['id']), [])]
                lbl = PREP / 'yolo' / 'labels' / ysplit / (Path(i['file_name']).stem + '.txt')
                lbl.parent.mkdir(parents=True, exist_ok=True)
                lbl.write_text('\n'.join(lines) + ('\n' if lines else ''))
        yaml_text = (f"path: {(PREP / 'yolo').as_posix()}\ntrain: images/train_p{p}\nval: images/valid\n"
                     f"test: images/test\nnames:\n" + ''.join(f'  {k}: {json.dumps(v)}\n' for k, v in names.items()))
        (PREP / 'yolo' / f'data_p{p}.yaml').write_text(yaml_text, encoding='utf-8')

    # statistik kelas
    rows = []
    for cid in real_ids:
        st = stats[cid]
        row = {'id': cid, 'nama': st['name']}
        for s in SPLITS:
            row[f'{s}_instance'] = st.get(f'{s}_inst', 0)
            row[f'{s}_gambar'] = len(st.get(f'{s}_img', ()))
        for p in PERCENTS[:-1]:
            ids_p = set(subsets[p])
            row[f'train_p{p}_instance'] = sum(1 for (s, iid), kept in good_anns.items() if s == 'train' and iid in ids_p
                                              for a, _ in kept if a['category_id'] == cid)
        row['median_luas_relatif'] = float(np.median(st['area_rel'])) if st['area_rel'] else None
        row['median_fill_ratio'] = float(np.median(st['fill'])) if st['fill'] else None
        ar, fr = row['median_luas_relatif'], row['median_fill_ratio']
        row['ukuran'] = None if ar is None else ('kecil' if ar < 0.01 else 'besar' if ar > 0.10 else 'sedang')
        row['bentuk'] = None if fr is None else ('tipis/berlubang' if fr < 0.5 else 'padat')
        row['langka_train'] = row['train_instance'] < 10
        rows.append(row)
    with open(PREP / 'statistik_kelas.csv', 'w', newline='', encoding='utf-8') as f:
        wr = csv.DictWriter(f, fieldnames=list(rows[0]))
        wr.writeheader()
        wr.writerows(rows)
    for p in PERCENTS:
        key = 'train_instance' if p == 100 else f'train_p{p}_instance'
        audit['split'][f'kelas_tanpa_instance_train_p{p}'] = [r['nama'] for r in rows if r[key] == 0]
    audit['kelas_tanpa_instance_test'] = [r['nama'] for r in rows if r['test_instance'] == 0]
    save_json([{'id': c['id'], 'name': c['name']} for c in real], PREP / 'classes.json', indent=1)
    save_json(audit, PREP / 'audit.json', indent=1)
    shutil.copy2(PREP / 'audit.json', HASIL / 'audit.json')
    shutil.copy2(PREP / 'statistik_kelas.csv', HASIL / 'statistik_kelas.csv')
    return audit


def load_classes():
    cls = load_json(PREP / 'classes.json')
    return [c['id'] for c in cls], [c['name'] for c in cls]


def show_audit():
    import pandas as pd
    audit = load_json(PREP / 'audit.json')
    print('Jumlah kelas nyata:', audit['jumlah_kelas'], '| root dibuang:', [c['name'] for c in audit['kategori_root_dibuang']])
    print(json.dumps(audit['split'], indent=1, ensure_ascii=False))
    print('Subset train:', audit['subset'])
    print('Duplikat identik antar-split:', len(audit['duplikat_identik_antar_split']))
    print('Gambar diperbaiki orientasinya:', audit.get('gambar_diperbaiki_orientasi'))
    print('Kelas tanpa instance di test:', audit['kelas_tanpa_instance_test'])
    print('Masalah (maks 20):', audit['masalah'][:20])
    df = pd.read_csv(PREP / 'statistik_kelas.csv')
    pd.set_option('display.max_rows', 200)
    print('\nTop 15 kelas terbanyak di train:')
    display(df.sort_values('train_instance', ascending=False).head(15))
    print('Kelas langka (<10 instance train):', df[df.langka_train].nama.tolist())
    print('Kelas kecil (median luas < 1% gambar):', df[df.ukuran == 'kecil'].nama.tolist())
    print('Kelas besar (median luas > 10%):', df[df.ukuran == 'besar'].nama.tolist())
    print('Kelas tipis/berlubang (fill ratio < 0.5):', df[df.bentuk == 'tipis/berlubang'].nama.tolist())
    return df

In [ ]:
if RUN_PREPARE:
    DATA_ROOT = get_dataset()
    print('Ekspor COCO ditemukan di', DATA_ROOT)
    counts = tuple(len(load_json(DATA_ROOT / split / '_annotations.coco.json')['images']) for split in SPLITS)
    if counts != (397, 166, 359):
        raise ValueError(f'Jumlah train/valid/test {counts} berbeda dari ekspor v5 yang diharapkan (397, 166, 359).')
    source = {
        'version': 5,
        'source': str(DATA_ROOT),
        'sha256_annotations': {
            split: hashlib.sha256((DATA_ROOT / split / '_annotations.coco.json').read_bytes()).hexdigest()
            for split in SPLITS
        },
    }
    provenance = PREP / 'dataset_source.json'
    if (PREP / 'audit.json').exists() and (not provenance.exists() or load_json(provenance) != source):
        raise RuntimeError('Cache data berasal dari sumber lain. Mulai sesi baru sebelum mengganti DATA_SOURCE.')
    audit = prepare(DATA_ROOT)
    if audit['masalah'] or audit['duplikat_identik_antar_split'] or audit['jumlah_kelas'] != 74:
        raise ValueError(f"Audit dataset gagal: {audit['masalah'][:10]}")
    save_json(source, provenance, indent=2)
    shutil.copy2(provenance, HASIL / provenance.name)
if (PREP / 'audit.json').exists():
    CLASS_IDS, CLASS_NAMES = load_classes()
    STAT_KELAS = show_audit()


Periksa audit dan pertahankan entri hipotesis bertanggal yang sudah ada di logbook. Catat kondisi aktual run laptop dan run Kaggle.

## 4. Evaluator bersama (COCO mask) untuk ketiga model
* pycocotools `segm`, maxDets 100, confidence minimum 0.001, mask biner threshold 0.5, skor skala 0–1.
* Prediksi dicocokkan ke ground truth **berdasarkan nama kelas**, sehingga evaluator yang sama dipakai juga untuk test set sendiri.
* Rata-rata hanya atas kelas yang punya ground truth pada set yang dievaluasi; kelas lain ditulis `null`.
* Waktu inferensi: 3 warm-up, `cuda.synchronize`, mencakup preprocessing + model + postprocessing mask; tidak termasuk baca file.
* Overlay memakai confidence ≥ 0.25 hanya untuk tampilan.
* Angka evaluator ini bisa berbeda dari metrik bawaan framework (uji data sintetis: YOLO 0,73 vs 0,81 Ultralytics).
  Laporan memakai evaluator bersama agar ketiga model diukur dengan aturan yang sama.
* Inferensi RF-DETR tidak dioptimasi (tanpa `optimize_for_inference`/TensorRT), sama seperti dua model lain.

In [ ]:
STAT_NAMES = ['mask_mAP50_95', 'mask_mAP50', 'mask_mAP75', 'mask_mAP_small', 'mask_mAP_medium', 'mask_mAP_large',
              'AR1', 'AR10', 'AR100', 'AR_small', 'AR_medium', 'AR_large']
PALETTE = [(230, 25, 75), (60, 180, 75), (255, 225, 25), (0, 130, 200), (245, 130, 48), (145, 30, 180),
           (70, 240, 240), (240, 50, 230), (210, 245, 60), (250, 190, 212), (0, 128, 128), (220, 190, 255)]


def sync():
    if CUDA:
        torch.cuda.synchronize()


def color_of(name):
    return PALETTE[sum(map(ord, name)) % len(PALETTE)]


def draw_masks(img, items, title=''):
    """items: list of (mask_bool, nama, skor_or_None)."""
    base = np.asarray(img).astype(np.float32).copy()
    for m, name, _ in items:
        base[m] = base[m] * 0.45 + np.array(color_of(name)) * 0.55
    out = Image.fromarray(base.astype(np.uint8))
    d = ImageDraw.Draw(out)
    for m, name, score in items:
        ys, xs = np.where(m)
        if len(xs):
            txt = name if score is None else f'{name} {score:.2f}'
            d.rectangle([xs.min(), ys.min(), xs.min() + 7 * len(txt), ys.min() + 12], fill=color_of(name))
            d.text((xs.min() + 1, ys.min()), txt, fill=(0, 0, 0))
    if title:
        d.rectangle([0, 0, 8 * len(title), 14], fill=(255, 255, 255))
        d.text((2, 1), title, fill=(0, 0, 0))
    return out


def gt_items(coco, iid):
    items = []
    for a in coco.imgToAnns[iid]:
        items.append((coco.annToMask(a).astype(bool), coco.cats[a['category_id']]['name'], None))
    return items


def side_by_side(images):
    w = sum(i.width for i in images)
    h = max(i.height for i in images)
    canvas = Image.new('RGB', (w, h), 'white')
    x = 0
    for i in images:
        canvas.paste(i, (x, 0))
        x += i.width
    return canvas


def evaluate(split_dir, predict, out_dir=None, img_ids=None, n_overlay=6, timing=True,
             save_predictions=True, conf_overlay=0.25, transform=None):
    split_dir = Path(split_dir)
    with quiet():
        gt = COCO(str(split_dir / '_annotations.coco.json'))
    name2id = {c['name']: c['id'] for c in gt.loadCats(gt.getCatIds())}
    ids = sorted(img_ids) if img_ids is not None else sorted(gt.getImgIds())
    eval_cats = [c for c in gt.getCatIds() if gt.getAnnIds(imgIds=ids, catIds=[c], iscrowd=False)]
    dets, times, overlays, unknown = [], [], [], set()
    if timing and ids:
        warm = load_rgb(split_dir / gt.imgs[ids[0]]['file_name'], gt.imgs[ids[0]])
        warm = transform(warm) if transform else warm
        for _ in range(3):
            predict(warm)
    for k, iid in enumerate(ids):
        info = gt.imgs[iid]
        img = load_rgb(split_dir / info['file_name'], info)
        if transform:
            img = transform(img)
        sync()
        t0 = time.perf_counter()
        preds = predict(img)
        sync()
        times.append(time.perf_counter() - t0)
        shown = []
        for p in preds:
            cid = name2id.get(p['name'])
            if cid is None:
                unknown.add(p['name'])
                continue
            m = np.asarray(p['mask']).astype(bool)
            if m.shape != (info['height'], info['width']):
                m = np.array(Image.fromarray(m.astype(np.uint8) * 255).resize((info['width'], info['height']),
                                                                              Image.NEAREST)) > 127
            rle = mask_util.encode(np.asfortranarray(m.astype(np.uint8)))
            rle = {'size': [int(v) for v in rle['size']], 'counts': rle['counts'].decode('ascii')}
            dets.append({'image_id': int(iid), 'category_id': int(cid), 'segmentation': rle, 'score': float(p['score'])})
            if k < n_overlay and p['score'] >= conf_overlay:
                shown.append((m, p['name'], float(p['score'])))
        if out_dir is not None and k < n_overlay:
            overlays.append((info['file_name'], side_by_side([draw_masks(img, gt_items(gt, iid), 'ground truth'),
                                                              draw_masks(img, shown, f'prediksi conf>={conf_overlay}')])))
    metrics = {'images': len(ids), 'detections': len(dets), 'kelas_dievaluasi': len(eval_cats),
               'score_threshold': 0.001, 'max_dets': 100, 'mask_threshold': 0.5,
               'ms_per_image_mean': 1000 * float(np.mean(times)) if times and timing else None,
               'ms_per_image_median': 1000 * float(np.median(times)) if times and timing else None,
               'prediksi_kelas_tidak_ada_di_gt': sorted(unknown)}
    per_class = {}
    summary_txt = ''
    if dets and eval_cats:
        with quiet():
            dt = gt.loadRes(dets)
        E = COCOeval(gt, dt, 'segm')
        E.params.imgIds, E.params.catIds = ids, eval_cats
        buf = io.StringIO()
        with contextlib.redirect_stdout(buf):
            E.evaluate()
            E.accumulate()
            E.summarize()
        summary_txt = buf.getvalue()
        for name, v in zip(STAT_NAMES, E.stats):
            metrics[name] = None if v < 0 else float(v)
        prec = E.eval['precision']  # [T, R, K, A, M]
        for k, cid in enumerate(E.params.catIds):
            p_all, p50 = prec[:, :, k, 0, -1], prec[0, :, k, 0, -1]
            per_class[gt.cats[cid]['name']] = {
                'AP50_95': float(p_all[p_all > -1].mean()) if (p_all > -1).any() else None,
                'AP50': float(p50[p50 > -1].mean()) if (p50 > -1).any() else None,
                'instances': len(gt.getAnnIds(imgIds=ids, catIds=[cid], iscrowd=False))}
    else:
        for name in STAT_NAMES:
            metrics[name] = 0.0 if eval_cats else None
    parents = {c.get('supercategory') for c in gt.cats.values()}
    for c in gt.getCatIds():
        if gt.cats[c]['name'] in parents and not gt.getAnnIds(catIds=[c]):
            continue   # kategori root sintetis Roboflow
        per_class.setdefault(gt.cats[c]['name'], {'AP50_95': None, 'AP50': None, 'instances': 0})
    metrics['per_class'] = per_class
    if out_dir is not None:
        out_dir = Path(out_dir)
        out_dir.mkdir(parents=True, exist_ok=True)
        save_json(metrics, out_dir / 'metrics.json', indent=1)
        (out_dir / 'coco_summary.txt').write_text(summary_txt)
        with open(out_dir / 'per_class.csv', 'w', newline='', encoding='utf-8') as f:
            wr = csv.writer(f)
            wr.writerow(['kelas', 'AP50_95', 'AP50', 'instances'])
            for n, v in sorted(per_class.items()):
                wr.writerow([n, v['AP50_95'], v['AP50'], v['instances']])
        if save_predictions:
            with gzip.open(out_dir / 'predictions.json.gz', 'wt', encoding='utf-8') as f:
                json.dump(dets, f, default=lambda o: o.tolist() if hasattr(o, 'tolist') else str(o))  # loadRes menambah field numpy
        for fname, im in overlays:
            im.save(out_dir / f'overlay_{Path(fname).stem}.jpg', quality=85)
    return metrics

## 5. Tiga model: training dan prediktor

In [ ]:
import torchvision
import torchvision.transforms.functional as TF
from torch.utils.data import Dataset, DataLoader
from torchvision.models.detection import maskrcnn_resnet50_fpn, MaskRCNN_ResNet50_FPN_Weights
from torchvision.models.detection.faster_rcnn import FastRCNNPredictor
from torchvision.models.detection.mask_rcnn import MaskRCNNPredictor
from collections import defaultdict


def seed_all(seed=SEED):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)


# ---------------- Mask R-CNN ----------------
class CocoMaskDataset(Dataset):
    def __init__(self, split_dir, class_ids, train=False):
        self.dir = Path(split_dir)
        with quiet():
            self.coco = COCO(str(self.dir / '_annotations.coco.json'))
        self.ids = sorted(self.coco.getImgIds())
        self.label = {cid: i + 1 for i, cid in enumerate(class_ids)}   # 0 = background
        self.train = train

    def __len__(self):
        return len(self.ids)

    def __getitem__(self, idx):
        info = self.coco.imgs[self.ids[idx]]
        h, w = info['height'], info['width']
        image = TF.to_tensor(load_rgb(self.dir / info['file_name'], info))
        masks, labels, boxes = [], [], []
        for a in self.coco.imgToAnns[self.ids[idx]]:
            if a.get('iscrowd', 0) or a['category_id'] not in self.label:
                continue
            m = self.coco.annToMask(a)
            ys, xs = np.where(m)
            if len(xs) == 0:
                continue
            masks.append(m)
            labels.append(self.label[a['category_id']])
            boxes.append([xs.min(), ys.min(), xs.max() + 1, ys.max() + 1])
        target = {'boxes': torch.tensor(boxes, dtype=torch.float32).reshape(-1, 4),
                  'labels': torch.tensor(labels, dtype=torch.int64),
                  'masks': torch.from_numpy(np.stack(masks)) if masks else torch.zeros((0, h, w), dtype=torch.uint8)}
        if self.train and random.random() < 0.5:
            image = image.flip(-1)
            target['masks'] = target['masks'].flip(-1)
            b = target['boxes'].clone()
            b[:, 0], b[:, 2] = w - target['boxes'][:, 2], w - target['boxes'][:, 0]
            target['boxes'] = b
        return image, target


def collate(batch):
    return tuple(zip(*batch))


def build_maskrcnn(num_classes, size, pretrained=True):
    if pretrained:
        m = maskrcnn_resnet50_fpn(weights=MaskRCNN_ResNet50_FPN_Weights.COCO_V1, min_size=size, max_size=size,
                                  box_detections_per_img=100, box_score_thresh=0.001)
        in_f = m.roi_heads.box_predictor.cls_score.in_features
        m.roi_heads.box_predictor = FastRCNNPredictor(in_f, num_classes)
        m.roi_heads.mask_predictor = MaskRCNNPredictor(256, 256, num_classes)
        return m
    return maskrcnn_resnet50_fpn(weights=None, weights_backbone=None, num_classes=num_classes, min_size=size,
                                 max_size=size, box_detections_per_img=100, box_score_thresh=0.001)


def maskrcnn_predict_fn(model, names, device):
    @torch.no_grad()
    def predict(img):
        out = model([TF.to_tensor(img).to(device)])[0]
        masks = (out['masks'][:, 0] > 0.5).cpu().numpy()
        return [{'name': names[int(l) - 1], 'score': float(s), 'mask': m}
                for l, s, m in zip(out['labels'].tolist(), out['scores'].tolist(), masks)]
    return predict


def train_maskrcnn(rd, p, epochs, cfg):
    seed_all()
    dev = torch.device('cuda' if CUDA else 'cpu')
    ds = CocoMaskDataset(PREP / 'coco' / f'p{p}' / 'train', CLASS_IDS, train=True)
    g = torch.Generator()
    g.manual_seed(SEED)
    dl = DataLoader(ds, batch_size=cfg['batch'], shuffle=True, num_workers=cfg['workers'], collate_fn=collate,
                    generator=g, persistent_workers=cfg['workers'] > 0)
    model = build_maskrcnn(len(CLASS_IDS) + 1, cfg['imgsz'], pretrained=True).to(dev)
    params = [q for q in model.parameters() if q.requires_grad]
    opt = torch.optim.SGD(params, lr=cfg['lr'], momentum=0.9, weight_decay=1e-4)
    total = epochs * len(dl)
    warm = min(cfg['warmup_iters'], max(1, total // 10))
    def lr_factor(it):
        if it < warm:
            return 0.001 + 0.999 * it / warm
        prog = (it - warm) / max(1, total - warm)
        return 0.01 + 0.99 * 0.5 * (1 + math.cos(math.pi * prog))
    sched = torch.optim.lr_scheduler.LambdaLR(opt, lr_factor)
    scaler = torch.amp.GradScaler('cuda', enabled=CUDA)
    fields = ['epoch', 'lr', 'loss', 'loss_classifier', 'loss_box_reg', 'loss_mask', 'loss_objectness',
              'loss_rpn_box_reg', 'train_seconds', 'val_mask_mAP50_95', 'val_mask_mAP50', 'val_seconds']
    log_path = rd / 'results.csv'
    with open(log_path, 'w', newline='') as f:
        csv.DictWriter(f, fieldnames=fields).writeheader()
    try:
        from torch.utils.tensorboard import SummaryWriter
        tb = SummaryWriter(str(rd / 'tensorboard'))
    except Exception:
        tb = None
    val_dir = PREP / 'coco' / f'p{p}' / 'valid'
    best, best_ep, skipped = -1.0, None, 0
    for ep in range(1, epochs + 1):
        model.train()
        t0 = time.time()
        sums, n = defaultdict(float), 0
        for imgs, tgts in dl:
            imgs = [i.to(dev) for i in imgs]
            tgts = [{k: v.to(dev) for k, v in t.items()} for t in tgts]
            with torch.autocast(dev.type, dtype=torch.float16, enabled=CUDA):
                ld = model(imgs, tgts)
                loss = sum(ld.values())
            if not torch.isfinite(loss):
                skipped += 1
                opt.zero_grad(set_to_none=True)
                continue
            opt.zero_grad(set_to_none=True)
            scaler.scale(loss).backward()
            scaler.step(opt)
            scale_before = scaler.get_scale()
            scaler.update()
            if scaler.get_scale() >= scale_before:   # langkah tidak dilewati AMP
                sched.step()
            sums['loss'] += loss.item()
            for k, v in ld.items():
                sums[k] += v.item()
            n += 1
        row = {'epoch': ep, 'lr': opt.param_groups[0]['lr'], 'train_seconds': round(time.time() - t0, 1),
               **{k: v / max(n, 1) for k, v in sums.items()}}
        ckpt = {'model': model.state_dict(), 'class_names': CLASS_NAMES, 'class_ids': CLASS_IDS,
                'imgsz': cfg['imgsz'], 'epoch': ep}
        if ep % cfg['val_every'] == 0 or ep == epochs:
            model.eval()
            t1 = time.time()
            m = evaluate(val_dir, maskrcnn_predict_fn(model, CLASS_NAMES, dev), timing=False, n_overlay=0,
                         save_predictions=False)
            row.update(val_mask_mAP50_95=m['mask_mAP50_95'], val_mask_mAP50=m['mask_mAP50'],
                       val_seconds=round(time.time() - t1, 1))
            if (m['mask_mAP50_95'] or 0) > best:
                best, best_ep = m['mask_mAP50_95'] or 0, ep
                torch.save({**ckpt, 'val_mask_mAP50_95': best}, rd / 'best.pt')
        torch.save(ckpt, rd / 'last.pt')
        with open(log_path, 'a', newline='') as f:
            csv.DictWriter(f, fieldnames=fields, restval='').writerow({k: row.get(k, '') for k in fields})
        if tb:
            for k, v in row.items():
                if k != 'epoch' and isinstance(v, (int, float)):
                    tb.add_scalar(k, v, ep)
        print(f"[maskrcnn p{p}] epoch {ep}/{epochs} loss {row.get('loss', float('nan')):.3f} "
              f"val mask mAP {row.get('val_mask_mAP50_95', '-')} ({row['train_seconds']}s)")
    if tb:
        tb.close()
    del model
    return {'checkpoint': str(rd / 'best.pt'), 'best_epoch': best_ep, 'val_mask_mAP50_95_seleksi': best,
            'epochs_completed': epochs, 'iterasi_nonfinite_dilewati': skipped, 'iterasi_per_epoch': len(dl),
            'log': str(log_path)}


def predictor_maskrcnn(ckpt_path):
    dev = torch.device('cuda' if CUDA else 'cpu')
    ck = torch.load(ckpt_path, map_location='cpu', weights_only=False)
    m = build_maskrcnn(len(ck['class_names']) + 1, ck['imgsz'], pretrained=False)
    m.load_state_dict(ck['model'])
    m.to(dev).eval()
    return maskrcnn_predict_fn(m, ck['class_names'], dev)


# ---------------- YOLO26s-seg ----------------
def train_yolo(rd, p, epochs, cfg):
    from ultralytics import YOLO
    state = {'best': -1.0, 'epoch': None}
    def on_model_save(trainer):   # dipanggil setelah last.pt disimpan
        m = (trainer.metrics or {}).get('metrics/mAP50-95(M)')
        if m is not None and m > state['best']:
            state['best'], state['epoch'] = float(m), int(trainer.epoch) + 1
            shutil.copy2(trainer.last, trainer.wdir / 'best_mask.pt')
    model = YOLO(cfg['weights'])
    model.add_callback('on_model_save', on_model_save)
    model.train(data=str(PREP / 'yolo' / f'data_p{p}.yaml'), epochs=epochs, imgsz=cfg['imgsz'], batch=cfg['batch'],
                seed=SEED, deterministic=True, workers=cfg['workers'], project=str(rd), name='ultralytics',
                exist_ok=True, patience=epochs + 1, plots=True, device=0 if CUDA else 'cpu', verbose=False)
    sd = rd / 'ultralytics'
    n_ep = max(0, sum(1 for _ in open(sd / 'results.csv')) - 1) if (sd / 'results.csv').exists() else None
    del model
    return {'checkpoint': str(sd / 'weights' / 'best_mask.pt'), 'best_epoch': state['epoch'],
            'val_mask_mAP50_95_seleksi': state['best'], 'catatan_seleksi': 'mask mAP50-95 validation Ultralytics',
            'epochs_completed': n_ep, 'iterasi_per_epoch': math.ceil(len(load_json(PREP / 'subset_ids.json')[f'p{p}']) / cfg['batch']),
            'log': str(sd / 'results.csv')}


def predictor_yolo(weights, imgsz=640):
    from ultralytics import YOLO
    m = YOLO(str(weights))
    def predict(img, conf=0.001):
        r = m.predict(img, imgsz=imgsz, conf=conf, max_det=100, retina_masks=True, verbose=False,
                      device=0 if CUDA else 'cpu')[0]
        if r.masks is None or len(r.boxes) == 0:
            return []
        masks = (r.masks.data > 0.5).cpu().numpy()
        return [{'name': r.names[int(c)], 'score': float(s), 'mask': mk}
                for c, s, mk in zip(r.boxes.cls.tolist(), r.boxes.conf.tolist(), masks)]
    return predict


# ---------------- RF-DETR-Seg Medium ----------------
def predictor_rfdetr(ckpt):
    from rfdetr import RFDETR
    with quiet():
        m = RFDETR.from_checkpoint(str(ckpt), trust_checkpoint=True)
    names = m.class_names
    def predict(img, conf=0.001):
        d = m.predict(img, threshold=conf)
        if d.mask is None or len(d) == 0:
            return []
        # class_id berbasis 0 pada class_names; slot terakhir = '__background__' (bukan objek) dan dibuang.
        cname = d.data.get('class_name')
        cname = cname if cname is not None else [names[c] if c < len(names) else '__background__' for c in d.class_id]
        keep = [i for i in np.argsort(-d.confidence) if cname[i] != '__background__'][:100]
        return [{'name': str(cname[i]), 'score': float(d.confidence[i]), 'mask': d.mask[i]} for i in keep]
    return predict


def train_rfdetr(rd, p, epochs, cfg):
    import rfdetr
    seed_all()
    Model = getattr(rfdetr, cfg['variant'])
    model = Model()
    out = rd / 'rfdetr'
    try:
        import tensorboard  # noqa: F401
        use_tb = True
    except ImportError:
        use_tb = False
    model.train(dataset_dir=str(PREP / 'coco' / f'p{p}'), epochs=epochs, batch_size=cfg['batch'],
                grad_accum_steps=cfg['grad_accum'], lr=cfg['lr'], output_dir=str(out), seed=SEED,
                checkpoint_interval=max(1, epochs), num_workers=cfg['workers'], tensorboard=use_tb,
                device='cuda' if CUDA else 'cpu')
    del model
    gc_cuda()
    cands = [out / n for n in ('checkpoint_best_total.pth', 'checkpoint_best_ema.pth', 'checkpoint_best_regular.pth')
             if (out / n).exists()]
    if not cands:
        raise FileNotFoundError('Checkpoint terbaik RF-DETR tidak ditemukan di ' + str(out))
    scores = {}
    for c in cands:   # seleksi dengan evaluator bersama pada VALIDATION
        scores[c.name] = evaluate(PREP / 'coco' / f'p{p}' / 'valid', predictor_rfdetr(c), timing=False,
                                  n_overlay=0, save_predictions=False)['mask_mAP50_95'] or 0.0
        gc_cuda()
    best = max(scores, key=scores.get)
    logs = sorted(out.rglob('metrics.csv'))
    n_ep = None
    if logs:
        import pandas as pd
        dfm = pd.read_csv(logs[0])
        n_ep = int(dfm['epoch'].max()) + 1 if 'epoch' in dfm else None
    n_img = len(load_json(PREP / 'subset_ids.json')[f'p{p}'])
    return {'checkpoint': str(out / best), 'val_mask_mAP50_95_seleksi': scores[best], 'kandidat_checkpoint': scores,
            'catatan_seleksi': 'evaluator bersama pada validation', 'epochs_completed': n_ep,
            'iterasi_per_epoch': math.ceil(n_img / cfg['batch']),
            'langkah_optimizer_per_epoch': math.ceil(n_img / (cfg['batch'] * cfg['grad_accum'])),
            'log': str(logs[0]) if logs else None}


def gc_cuda():
    import gc
    gc.collect()
    if CUDA:
        torch.cuda.empty_cache()


TRAINERS = {'maskrcnn': train_maskrcnn, 'yolo': train_yolo, 'rfdetr': train_rfdetr}


def predictor_for(run):
    kind, ck = run['model'], run['checkpoint']
    if kind == 'maskrcnn':
        return predictor_maskrcnn(ck)
    if kind == 'yolo':
        return predictor_yolo(ck, MODEL_CFG['yolo']['imgsz'])
    return predictor_rfdetr(ck)

## 6. Manajemen run: catatan waktu GPU, konfigurasi, dan evaluasi otomatis

In [ ]:
def run_dir_for(model, p, tag):
    return RUNS / f'{model}_p{p}_{tag}'


def localize(run, rd):
    """Perbaiki path checkpoint bila run disalin dari sesi Kaggle lain."""
    rd = Path(rd)
    relative = run.get('checkpoint_relative')
    if relative:
        ck = rd / relative
    else:
        name = run['checkpoint'].replace(chr(92), '/').rsplit('/', 1)[-1]
        ck = next(rd.rglob(name), rd / name)
    if not ck.is_file():
        raise FileNotFoundError(f'Checkpoint tidak ditemukan di {rd}: {ck.name}')
    return dict(run, checkpoint=str(ck))


def is_t4_run(run):
    env = run.get('lingkungan', {})
    return (run.get('protocol') == RUN_PROTOCOL and env.get('kaggle') is True
            and 'T4' in env.get('gpu', '').split())


def train_run(model, p, epochs, tag, eval_splits=('valid', 'test')):
    if not HIPOTESIS_SUDAH_DICOMMIT:
        raise RuntimeError('Tulis dan commit hipotesis di LOGBOOK.md dulu, lalu set HIPOTESIS_SUDAH_DICOMMIT = True.')
    source_path = PREP / 'dataset_source.json'
    if not source_path.exists():
        raise RuntimeError('Aktifkan RUN_PREPARE sebelum training agar data v5 diaudit.')
    source = load_json(source_path)['sha256_annotations']
    rd = run_dir_for(model, p, tag)
    if (rd / 'run.json').exists() and load_json(rd / 'run.json').get('status') == 'selesai':
        previous = load_json(rd / 'run.json')
        if (not is_t4_run(previous) or previous.get('dataset_hashes') != source
                or previous['epochs'] != epochs or previous['seed'] != SEED
                or previous['config'] != MODEL_CFG[model]):
            raise RuntimeError(f'Konfigurasi berubah untuk {rd.name}. Mulai sesi baru atau gunakan TAG baru.')
        localize(previous, rd)
        print('Lewati training (sudah selesai):', rd.name)
        for split in eval_splits:
            if not (rd / f'eval_{split}' / 'metrics.json').exists():
                evaluate_run(rd, split)
        return load_json(rd / 'run.json')
    if rd.exists():
        rd.rename(rd.with_name(rd.name + '_gagal_' + datetime.datetime.now().strftime('%Y%m%d_%H%M%S')))
    rd.mkdir(parents=True)
    cfg = MODEL_CFG[model]
    run = {'model': model, 'percent': p, 'tag': tag, 'epochs': epochs, 'seed': SEED, 'nrp': NRP, 'config': cfg,
           'n_train_images': len(load_json(PREP / 'subset_ids.json')[f'p{p}']), 'lingkungan': ENV,
           'mulai': datetime.datetime.now().isoformat(timespec='seconds'), 'status': 'berjalan'}
    run.update(protocol=RUN_PROTOCOL, dataset_hashes=source)
    save_json(run, rd / 'run.json', indent=1)
    if CUDA:
        torch.cuda.reset_peak_memory_stats()
    t0 = time.time()
    try:
        info = TRAINERS[model](rd, p, epochs, cfg)
    except Exception as e:
        run.update(status='gagal', error=repr(e), elapsed_seconds=round(time.time() - t0, 1))
        save_json(run, rd / 'run.json', indent=1)
        gc_cuda()
        raise
    run.update(info)
    run["checkpoint_relative"] = Path(run["checkpoint"]).relative_to(rd).as_posix()
    if HAPUS_CHECKPOINT_LAIN:
        run['checkpoint_dihapus_mb'] = cleanup_checkpoints(rd, run['checkpoint'])
    run.update(status='selesai', selesai=datetime.datetime.now().isoformat(timespec='seconds'),
               elapsed_seconds=round(time.time() - t0, 1),
               vram_puncak_gb=round(torch.cuda.max_memory_allocated() / 1e9, 2) if CUDA else None)
    save_json(run, rd / 'run.json', indent=1)
    log = HASIL / 'catatan_waktu_gpu.csv'
    new = not log.exists()
    with open(log, 'a', newline='') as f:
        wr = csv.writer(f)
        if new:
            wr.writerow(['run', 'gpu', 'mulai', 'selesai', 'menit', 'epoch_selesai', 'batch', 'vram_puncak_gb'])
        wr.writerow([rd.name, GPU_NAME, run['mulai'], run['selesai'], round(run['elapsed_seconds'] / 60, 1),
                     run.get('epochs_completed'), cfg.get('batch'), run['vram_puncak_gb']])
    gc_cuda()
    for split in eval_splits:
        evaluate_run(rd, split)
    return run


def cleanup_checkpoints(rd, keep):
    keep = Path(keep).resolve()
    freed = 0
    for f in Path(rd).rglob('*'):
        if f.is_file() and f.suffix in ('.pt', '.pth', '.ckpt') and f.resolve() != keep:
            freed += f.stat().st_size
            f.unlink()
    return round(freed / 1e6, 1)


def evaluate_run(rd, split='test', split_dir=None, name=None, **kw):
    rd = Path(rd)
    run = localize(load_json(rd / 'run.json'), rd)
    split_dir = Path(split_dir) if split_dir else PREP / 'coco' / f'p{run["percent"]}' / split
    out = rd / f'eval_{name or split}'
    m = evaluate(split_dir, predictor_for(run), out_dir=out, **kw)
    gc_cuda()
    print(f"{rd.name} [{name or split}] mask mAP50-95={m['mask_mAP50_95']} mAP50={m['mask_mAP50']} "
          f"{m['ms_per_image_median'] or 0:.1f} ms/gambar")
    return m


def write_zip(name, files):
    path = WORK / f'{name}.zip'
    tmp = path.with_suffix('.zip.part')
    with zipfile.ZipFile(tmp, 'w', compression=zipfile.ZIP_STORED) as archive:
        for file in files:
            archive.write(file, file.relative_to(WORK).as_posix())
    tmp.replace(path)
    print(f'Unduh dari Kaggle Output: {path.name} ({path.stat().st_size / 1e6:.1f} MB)')
    return path


def export_light(zip_name='hasil_ringan', run_dirs=()):
    excluded = {'.pt', '.pth', '.ckpt', '.cache'}
    files = [f for f in HASIL.rglob('*') if f.is_file() and f.suffix not in excluded]
    for rd in run_dirs:
        files.extend(f for f in rd.rglob('*') if f.is_file() and f.suffix not in excluded)
    return write_zip(zip_name, files)


## 7. Smoke test (2 epoch, 25% data) — bukan hasil eksperimen
Periksa: loss turun, tidak OOM, perkiraan waktu per epoch. Jika OOM, kecilkan batch di sel 1 dan ulangi smoke test.
Catat waktu dan keputusan di logbook.

In [ ]:
if RUN_SMOKE:
    estimasi = {}
    for model in TRAIN_MODELS:
        r = train_run(model, 25, SMOKE_EPOCHS, 'smoke', eval_splits=('valid',))
        per_epoch = r['elapsed_seconds'] / max(1, SMOKE_EPOCHS)
        # batas atas kasar: waktu per epoch naik sebanding jumlah data (25% → 50% → 100% = 1 + 2 + 4)
        estimasi[model] = per_epoch * EPOCHS * 7 / 60
        print(f"{model}: {r['elapsed_seconds'] / 60:.1f} menit untuk {SMOKE_EPOCHS} epoch, "
              f"VRAM puncak {r['vram_puncak_gb']} GB → perkiraan 3 run x {EPOCHS} epoch sekitar {estimasi[model]:.0f} menit")
    print(f'Perkiraan kasar total 9 run {sum(estimasi.values()) / 60:.1f} jam (termasuk pemuatan model dan validasi tiap epoch)')
    save_json(estimasi, HASIL / 'estimasi_waktu_menit.json', indent=1)
    export_light('hasil_smoke_t4', [run_dir_for(model, 25, 'smoke') for model in TRAIN_MODELS])

## 8. Sembilan training utama (3 model × 25/50/100%)
Setiap run mulai dari bobot pretrained COCO yang baru. Validation untuk memilih checkpoint; test dievaluasi sekali
setelah run selesai. Untuk sesi Kaggle 12 jam, jalankan per model (ubah `TRAIN_MODELS`) dengan **Save Version → Save & Run All**.

In [ ]:
if RUN_TRAIN:
    for model in TRAIN_MODELS:
        for p in TRAIN_PERCENTS:
            train_run(model, p, EPOCHS, TAG)

## 9. Tabel dan grafik hasil (eksperimen NRP)
Mengumpulkan run dari folder kerja **dan** dari output sesi Kaggle sebelumnya yang ditambahkan sebagai input.

In [ ]:
def collect_previous_runs():
    if not IN_KAGGLE:
        return
    bases = [Path(p) for p in PREVIOUS_RUN_INPUTS] or [Path('/kaggle/input')]
    for base in bases:
        if not base.is_dir():
            raise FileNotFoundError(f'Input hasil Kaggle tidak ditemukan: {base}')
        for rj in sorted(base.rglob('runs/*/run.json')):
            run = load_json(rj)
            src = rj.parent
            if (run.get('tag') != TAG or run.get('status') != 'selesai' or not is_t4_run(run)
                    or not (src / 'eval_test' / 'metrics.json').is_file()):
                continue
            localize(run, src)
            dst = RUNS / src.name
            if dst.exists():
                if load_json(dst / 'run.json') != run:
                    raise ValueError(f'Dua run berbeda memakai nama {src.name}. Pilih input secara eksplisit.')
                continue
            shutil.copytree(src, dst)
            print('Disalin dari input', src)


def report_runs(require_all=False):
    runs = {}
    signatures = set()
    by_model = {}
    for rd in sorted(RUNS.glob(f'*_{TAG}')):
        if not (rd / 'run.json').is_file() or not (rd / 'eval_test' / 'metrics.json').is_file():
            continue
        run = load_json(rd / 'run.json')
        if run.get('status') != 'selesai' or not is_t4_run(run):
            raise ValueError(f'Run tanpa bukti Kaggle T4 yang sesuai: {rd.name}.')
        localize(run, rd)
        key = (run['model'], run['percent'])
        if key in runs:
            raise ValueError(f'Run duplikat: {key}')
        runs[key] = rd
        signatures.add(json.dumps([run['dataset_hashes'], run['seed'], run['epochs']], sort_keys=True))
        config = run['config']
        if run['model'] in by_model and by_model[run['model']] != config:
            raise ValueError(f'Konfigurasi {run["model"]} berbeda antarsubset.')
        by_model[run['model']] = config
    if len(signatures) > 1:
        raise ValueError('Dataset, seed, atau jumlah epoch berbeda antar-run.')
    expected = {(m, p) for m in MODEL_CFG for p in PERCENTS}
    missing = expected - runs.keys()
    if missing:
        print('Run yang masih diperlukan:', sorted(missing))
        if require_all:
            raise ValueError('Tambahkan output Kaggle yang mencakup sembilan run.')
    return list(runs.values())


def rebuild_gpu_log(run_dirs):
    with (HASIL / 'catatan_waktu_gpu.csv').open('w', newline='', encoding='utf-8') as file:
        writer = csv.writer(file)
        writer.writerow(['run', 'gpu', 'mulai', 'selesai', 'menit', 'epoch_selesai', 'batch', 'vram_puncak_gb'])
        for rd in run_dirs:
            r = load_json(rd / 'run.json')
            writer.writerow([rd.name, r['lingkungan']['gpu'], r['mulai'], r['selesai'],
                             round(r['elapsed_seconds'] / 60, 1), r.get('epochs_completed'),
                             r['config']['batch'], r['vram_puncak_gb']])


def results_table(tag=TAG, split='test'):
    import pandas as pd
    rows = []
    for rj in sorted(RUNS.glob(f'*_{tag}/run.json')):
        run = load_json(rj)
        mp = rj.parent / f'eval_{split}' / 'metrics.json'
        if run.get('status') != 'selesai' or not mp.exists():
            continue
        m = load_json(mp)
        rows.append({'model': run['model'], 'train_%': run['percent'], 'n_train': run['n_train_images'],
                     'mask_mAP50_95': m['mask_mAP50_95'], 'mask_mAP50': m['mask_mAP50'], 'mask_mAP75': m['mask_mAP75'],
                     'mAP_small': m['mask_mAP_small'], 'mAP_medium': m['mask_mAP_medium'],
                     'mAP_large': m['mask_mAP_large'], 'ms_per_gambar': m['ms_per_image_median'],
                     'epoch_selesai': run.get('epochs_completed'), 'best_epoch': run.get('best_epoch'),
                     'menit_gpu': round(run['elapsed_seconds'] / 60, 1), 'gpu': run['lingkungan']['gpu']})
    return pd.DataFrame(rows).sort_values(['model', 'train_%']) if rows else pd.DataFrame()


def plot_nrp(df, path):
    import matplotlib.pyplot as plt
    fig, axes = plt.subplots(1, 2, figsize=(11, 4))
    for model, g in df.groupby('model'):
        axes[0].plot(g['train_%'], g['mask_mAP50_95'], marker='o', label=model)
        axes[1].plot(g['train_%'], g['mask_mAP50'], marker='o', label=model)
    for ax, t in zip(axes, ['Mask mAP@50–95 (test asli)', 'Mask mAP@50 (test asli)']):
        ax.set_xticks(PERCENTS)
        ax.set_xlabel('Data train (%)')
        ax.set_title(t)
        ax.grid(alpha=.3)
        ax.legend()
    fig.suptitle(f'Eksperimen NRP {NRP}: pengaruh jumlah data train')
    fig.tight_layout()
    fig.savefig(path, dpi=200)
    fig.savefig(path.with_suffix('.pdf'))
    plt.show()


def per_class_table(percent=100, tag=TAG, split='test'):
    import pandas as pd
    cols = {}
    for model in ('maskrcnn', 'yolo', 'rfdetr'):
        mp = run_dir_for(model, percent, tag) / f'eval_{split}' / 'metrics.json'
        if mp.exists():
            pc = load_json(mp)['per_class']
            cols[model] = {k: v['AP50_95'] for k, v in pc.items()}
            cols['instances'] = {k: v['instances'] for k, v in pc.items()}
    return pd.DataFrame(cols)


def export_bobot(run_dirs, zip_name):
    files = []
    for rd in run_dirs:
        rj = rd / 'run.json'
        run = localize(load_json(rj), rd)
        files.extend([rj, Path(run['checkpoint']), *rd.glob('eval_*/metrics.json')])
    return write_zip(zip_name, files)


def export_report(run_dirs, label):
    rebuild_gpu_log(run_dirs)
    for split, filename in [('test', 'tabel_hasil_nrp.csv'), ('valid', 'tabel_validasi_nrp.csv')]:
        df = results_table(split=split)
        if not df.empty:
            df.to_csv(HASIL / filename, index=False)
            if split == 'test':
                display(df)
                plot_nrp(df, HASIL / 'grafik_nrp.png')
    pc = per_class_table(100)
    if not pc.empty:
        pc.to_csv(HASIL / 'ap_per_kelas_p100.csv', index_label='kelas')
    export_light(f'hasil_{label}', run_dirs)
    export_bobot(run_dirs, f'bobot_{label}')


if RUN_SUMMARY:
    collect_previous_runs()
if RUN_TRAIN or RUN_SUMMARY:
    run_dirs = report_runs(require_all=RUN_SUMMARY)
    if run_dirs:
        label = TAG if len(run_dirs) == len(MODEL_CFG) * len(PERCENTS) else TAG + '_parsial'
        export_report(run_dirs, label)


## 10. Test set foto HP sendiri dan domain shift
1. Ekspor proyek Roboflow foto sendiri sebagai **COCO Segmentation**; nama kelas harus sama persis dengan v5.
2. Isi `OWN_TEST_DIR` dengan folder yang berisi `_annotations.coco.json` dan gambar.
3. `KONDISI_SULIT` = daftar nama file asli (mis. `IMG_1234.jpg`) yang termasuk kondisi sulit.
4. Checkpoint yang dipakai sama dengan evaluasi test asli. Tidak ada training ulang.

In [ ]:
OWN_TEST_DIR = ''            # mis. '/kaggle/input/test-sendiri/test'
KONDISI_SULIT = []           # mis. ['IMG_1234.jpg', 'IMG_1240.jpg']
OWN_PERCENT = 100
OWN_MIN_PHOTOS = 36          # jumlah yang disetujui dosen untuk tugas ini


def own_ids(split_dir, names):
    with quiet():
        gt = COCO(str(Path(split_dir) / '_annotations.coco.json'))
    keys = {n.replace('.', '_').lower(): n for n in names} | {Path(n).stem.lower(): n for n in names}
    found, ids = set(), []
    for iid, info in gt.imgs.items():
        fn = Path(info['file_name']).name.split('.rf.')[0].lower()
        stem = Path(fn).stem
        orig = str(info.get('extra', {}).get('name', '')).lower()
        for k, n in keys.items():
            if fn == k or stem == k or fn.startswith(k + '_') or orig == n.lower():
                ids.append(iid)
                found.add(n)
                break
    missing = sorted(set(names) - found)
    if missing:
        print('PERINGATAN: tidak ditemukan di ekspor:', missing)
    return ids


def export_exif(folder, out_csv=HASIL / 'exif_test_sendiri.csv'):
    """Jalankan pada FOTO ASLI dari HP (bukan hasil ekspor Roboflow)."""
    try:
        import pillow_heif
        pillow_heif.register_heif_opener()
    except ImportError:
        pass
    tags = {271: 'Make', 272: 'Model', 306: 'DateTime', 274: 'Orientation'}
    sub = {36867: 'DateTimeOriginal', 33434: 'ExposureTime', 33437: 'FNumber', 34855: 'ISO', 37386: 'FocalLength'}
    rows = []
    for f in sorted(Path(folder).iterdir()):
        if f.suffix.lower() not in ('.jpg', '.jpeg', '.png', '.heic', '.heif'):
            continue
        im = Image.open(f)
        ex = im.getexif()
        row = {'file': f.name, 'width': im.width, 'height': im.height}
        row.update({v: str(ex.get(k, '')) for k, v in tags.items()})
        ifd = ex.get_ifd(0x8769)
        row.update({v: str(ifd.get(k, '')) for k, v in sub.items()})
        rows.append(row)
    with open(out_csv, 'w', newline='', encoding='utf-8') as fh:
        wr = csv.DictWriter(fh, fieldnames=list(rows[0]))
        wr.writeheader()
        wr.writerows(rows)
    print(len(rows), 'foto →', out_csv)
    return rows


if RUN_OWN_TEST:
    collect_previous_runs()
    run_dirs = report_runs(require_all=True)
    if not OWN_TEST_DIR:
        raise ValueError('Isi OWN_TEST_DIR dengan folder foto HP yang berisi _annotations.coco.json.')
    own_data = load_json(Path(OWN_TEST_DIR) / '_annotations.coco.json')
    if len(own_data['images']) < OWN_MIN_PHOTOS:
        raise ValueError(f'Test pribadi perlu minimal {OWN_MIN_PHOTOS} foto.')
    import pandas as pd
    rows = []
    hard_ids = own_ids(OWN_TEST_DIR, KONDISI_SULIT) if KONDISI_SULIT else None
    if len(set(KONDISI_SULIT)) < 10 or len(hard_ids or []) != len(set(KONDISI_SULIT)):
        raise ValueError('Isi minimal 10 nama foto sulit yang masing-masing cocok dengan satu foto di ekspor COCO.')
    for model in ('maskrcnn', 'yolo', 'rfdetr'):
        rd = run_dir_for(model, OWN_PERCENT, TAG)
        if not (rd / 'run.json').exists():
            print('Run tidak ada:', rd.name)
            continue
        m_own = evaluate_run(rd, split_dir=OWN_TEST_DIR, name='test_sendiri', n_overlay=50)
        m_hard = evaluate_run(rd, split_dir=OWN_TEST_DIR, name='test_sendiri_sulit', img_ids=hard_ids,
                              n_overlay=0) if hard_ids else {}
        m_asli = load_json(rd / 'eval_test' / 'metrics.json')
        rows.append({'model': model, 'mAP50_95_test_asli': m_asli['mask_mAP50_95'],
                     'mAP50_test_asli': m_asli['mask_mAP50'],
                     'mAP50_test_sendiri': m_own['mask_mAP50'],
                     'foto_sendiri': len(own_data['images']), 'foto_sulit': len(hard_ids),
                     'mAP50_95_test_sendiri': m_own['mask_mAP50_95'],
                     'selisih_poin': None if m_own['mask_mAP50_95'] is None else
                     round(100 * (m_asli['mask_mAP50_95'] - m_own['mask_mAP50_95']), 1),
                     'mAP50_95_kondisi_sulit': m_hard.get('mask_mAP50_95'),
                     'kelas_dievaluasi_sendiri': m_own['kelas_dievaluasi']})
    dshift = pd.DataFrame(rows)
    dshift.to_csv(HASIL / 'domain_shift.csv', index=False)
    display(dshift)
    export_light(f'hasil_test_pribadi_{TAG}', run_dirs)

## 11. Analisis kegagalan: studi kasus dan eksperimen kecil
`tampilkan_kasus` membuat gambar GT + prediksi ketiga model untuk satu foto. `uji_variasi` menjalankan eksperimen kecil
pada foto yang sama dengan **checkpoint tetap** dan satu variabel diubah (resolusi inferensi, gamma/kecerahan, threshold),
lalu menghitung mask AP per gambar.

In [ ]:
_PRED_CACHE = {}


def get_predictor(model, percent=100, tag=TAG):
    key = (model, percent, tag)
    if key not in _PRED_CACHE:
        rd = run_dir_for(model, percent, tag)
        if not (rd / 'run.json').exists():
            print('Run belum ada, dilewati:', rd.name)
            return None
        _PRED_CACHE[key] = predictor_for(localize(load_json(rd / 'run.json'), rd))
    return _PRED_CACHE[key]


def tampilkan_kasus(split_dir, file_name, conf=0.25, percent=100, simpan=True):
    split_dir = Path(split_dir)
    with quiet():
        gt = COCO(str(split_dir / '_annotations.coco.json'))
    info = next(i for i in gt.imgs.values() if i['file_name'] == file_name)
    img = load_rgb(split_dir / file_name, info)
    panels = [draw_masks(img, gt_items(gt, info['id']), 'ground truth')]
    for model in ('maskrcnn', 'yolo', 'rfdetr'):
        pred = get_predictor(model, percent)
        if pred is None:
            continue
        preds = [p for p in pred(img) if p['score'] >= conf]
        panels.append(draw_masks(img, [(np.asarray(p['mask']).astype(bool), p['name'], p['score']) for p in preds],
                                 f'{model} conf>={conf}'))
    out = side_by_side(panels)
    if simpan:
        (HASIL / 'kasus').mkdir(exist_ok=True)
        out.save(HASIL / 'kasus' / f'kasus_{Path(file_name).stem}.jpg', quality=90)
    display(out)
    return out


def adjust_gamma(gamma):
    def f(img):
        arr = np.asarray(img).astype(np.float32) / 255.0
        return Image.fromarray((np.clip(arr ** gamma, 0, 1) * 255).astype(np.uint8))
    return f


def uji_variasi(split_dir, file_name, model, variasi, percent=100):
    """variasi: dict nama → dict(transform=fungsi_gambar | None, imgsz=int | None).
    Contoh: {'asli': {}, 'gamma 0.6': {'transform': adjust_gamma(0.6)}, 'imgsz 960': {'imgsz': 960}}"""
    import pandas as pd
    split_dir = Path(split_dir)
    with quiet():
        gt = COCO(str(split_dir / '_annotations.coco.json'))
    iid = next(i['id'] for i in gt.imgs.values() if i['file_name'] == file_name)
    rd = run_dir_for(model, percent, TAG)
    run = localize(load_json(rd / 'run.json'), rd)
    rows = []
    for nama, v in variasi.items():
        if v.get('imgsz') and model == 'yolo':
            pred = predictor_yolo(run['checkpoint'], v['imgsz'])
        elif v.get('imgsz'):
            raise ValueError('imgsz variasi saat ini hanya untuk YOLO; untuk model lain ubah ukuran gambar lewat transform')
        else:
            pred = get_predictor(model, percent)
        m = evaluate(split_dir, pred, img_ids=[iid], timing=False, n_overlay=0, save_predictions=False,
                     transform=v.get('transform'))
        rows.append({'variasi': nama, 'mask_AP50_95_gambar_ini': m['mask_mAP50_95'], 'mask_AP50': m['mask_mAP50']})
    df = pd.DataFrame(rows)
    display(df)
    return df


# Contoh pemakaian (aktifkan setelah test sendiri tersedia):
# tampilkan_kasus(OWN_TEST_DIR, 'IMG_1234_jpg.rf.xxxx.jpg')
# uji_variasi(OWN_TEST_DIR, 'IMG_1234_jpg.rf.xxxx.jpg', 'yolo',
#             {'asli': {}, 'imgsz 960': {'imgsz': 960}, 'imgsz 1280': {'imgsz': 1280}})
# uji_variasi(OWN_TEST_DIR, 'IMG_1234_jpg.rf.xxxx.jpg', 'rfdetr',
#             {'asli': {}, 'gamma 0.5 (dicerahkan)': {'transform': adjust_gamma(0.5)}})

## 12. Demo langsung (10 menit)
1. Foto baru saat sesi → salin ke laptop → isi `DEMO_FOTO` → jalankan sel.
2. "Ganti threshold" → ubah `DEMO_CONF` lalu jalankan ulang.
3. "Tampilkan AP satu kelas" → `ap_kelas('nama_kelas')`.

In [ ]:
DEMO_FOTO = ''
DEMO_CONF = 0.25
DEMO_PERCENT = 100


def demo(foto=DEMO_FOTO, conf=DEMO_CONF, percent=DEMO_PERCENT):
    img = load_rgb(foto)
    panels = []
    for model in ('maskrcnn', 'yolo', 'rfdetr'):
        pred = get_predictor(model, percent)
        if pred is None:
            continue
        sync()
        t0 = time.perf_counter()
        preds = pred(img)
        sync()
        ms = 1000 * (time.perf_counter() - t0)
        shown = [p for p in preds if p['score'] >= conf]
        print(f'{model}: {len(shown)} objek (conf ≥ {conf}), {ms:.0f} ms →',
              ', '.join(f"{p['name']} {p['score']:.2f}" for p in shown[:12]))
        panels.append(draw_masks(img, [(np.asarray(p['mask']).astype(bool), p['name'], p['score']) for p in shown],
                                 f'{model} conf>={conf}'))
    out = side_by_side(panels)
    display(out)
    return out


def ap_kelas(nama, percent=100, split='test'):
    for model in ('maskrcnn', 'yolo', 'rfdetr'):
        mp = run_dir_for(model, percent, TAG) / f'eval_{split}' / 'metrics.json'
        if mp.exists():
            v = load_json(mp)['per_class'].get(nama)
            print(f'{model:9s} {nama}: AP50-95={v and v["AP50_95"]}  AP50={v and v["AP50"]}  '
                  f'instance={v and v["instances"]}')


if RUN_DEMO and DEMO_FOTO:
    collect_previous_runs()
    demo()